# Drilling advisory model — final near_5 + LightGBM + pruned features

Финальная версия второго notebook.

Что делает notebook:

1. загружает `united_rock_energy_segment_quantile.csv` из первого notebook;
2. восстанавливает признаки, необходимые для `near_5` моделей;
3. использует один устойчивый признак энергоёмкости `hardness_score_smooth` и категорию `rock_energy_type_final`;
4. обучает `rotation_model_near5` и `speed_model_near5`;
5. выполняет offline replay оптимизатора;
6. сохраняет simulator-ready artifacts.

По итогам feature-pruning удалены multiscale-hardness признаки (`hardness_score`, `hardness_score_smooth_12`, `hardness_score_smooth_30`) и relative-diff признаки (`*_rel_diff1`).
В модели не используются residual-признаки и обратные величины к proxy-MSE.


In [1]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

try:
    import lightgbm as lgb
except Exception as exc:
    raise ImportError(
        "The final advisory notebook requires lightgbm. "
        "Install it with: pip install lightgbm"
    ) from exc

import joblib

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import (
    LABELED_DATA_PATH,
    DRILLING_ADVISORY_ARTIFACT_DIR,
    DRILLING_ADVISORY_REPORT_DIR,
    RANDOM_STATE,
    EPS,
    TARGET_HORIZON,
    GRID_SIZE,
    MAX_DELTA_FRAC,
    FINAL_OPTIMIZER_MODE,
    CHANGE_PENALTY_WEIGHT,
    BOUNDARY_PENALTY_WEIGHT,
    BOUNDARY_START,
    ensure_output_dirs,
)

ensure_output_dirs()

DATA_PATH = LABELED_DATA_PATH
ARTIFACT_DIR = DRILLING_ADVISORY_ARTIFACT_DIR
REPORT_DIR = DRILLING_ADVISORY_REPORT_DIR

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## 1. Load prepared rock-energy dataset

In [2]:
df = pd.read_csv(DATA_PATH).drop(columns=["Unnamed: 0"], errors="ignore")
df["processing_time"] = pd.to_datetime(df["processing_time"], errors="raise")
df = df.sort_values(["well_id", "processing_time"]).reset_index(drop=True)
df["rock_energy_type_final"] = df["rock_energy_type_final"].fillna("unknown").astype(str)

required_cols = [
    "processing_time",
    "well_id",
    "pressure_axis",
    "pressure_rotation",
    "rotation",
    "speed",
    "hardness_score_smooth",
    "rock_energy_type_final",
]

print("Loaded:", DATA_PATH)
print("Shape:", df.shape)

display(df[required_cols].head())
display(df[["pressure_axis", "pressure_rotation", "rotation", "speed", "hardness_score_smooth"]].describe(percentiles=[.01, .05, .5, .95, .99]))


Loaded: /home/alex/Desktop/OptimalDrilling/notebooks/united_rock_energy_segment_quantile.csv
Shape: (415049, 87)


,processing_time,well_id,pressure_axis,pressure_rotation,rotation,speed,hardness_score_smooth,rock_energy_type_final
0,2025-08-24 10:09:50.980,19601,804,4551,74.256,0.002755,NaN,unknown
1,2025-08-24 10:10:00.260,19601,763,3782,73.812,0.003030,NaN,unknown
2,2025-08-24 10:10:05.199,19601,879,4407,73.512,0.006060,NaN,unknown
3,2025-08-24 10:10:14.610,19601,721,3705,73.962,0.002755,NaN,unknown
4,2025-08-24 10:10:34.197,19601,859,3883,74.256,0.001515,NaN,unknown


,pressure_axis,pressure_rotation,rotation,speed,hardness_score_smooth
count,415049.000000,415049.000000,415049.000000,415049.000000,3.826350e+05
mean,17473.667273,14134.146325,103.945315,0.013116,2.101202e-15
std,4682.982816,3243.523440,13.370361,0.006615,9.999975e-01
min,317.000000,784.000000,50.010000,0.001002,-6.036494e+00
1%,3713.000000,6271.000000,64.980000,0.002755,-2.698224e+00
5%,6645.000000,8246.000000,81.750000,0.005050,-1.737642e+00
50%,18861.000000,14637.000000,103.158000,0.012120,8.955322e-02
95%,22343.000000,18758.600000,138.474000,0.024240,1.660707e+00
99%,23626.000000,20881.000000,139.020000,0.030300,2.116441e+00
max,24872.000000,26318.000000,139.578000,0.038957,3.723870e+00


## 2. Feature engineering

In [3]:
def add_features(data):
    out = data.copy()

    out["total_pressure"] = out["pressure_axis"] + out["pressure_rotation"]
    out["pressure_balance"] = out["pressure_axis"] / (out["total_pressure"] + EPS)
    out["axis_over_rot_pressure"] = out["pressure_axis"] / (out["pressure_rotation"] + EPS)
    out["rot_pressure_over_axis"] = out["pressure_rotation"] / (out["pressure_axis"] + EPS)
    out["rotation_efficiency"] = out["rotation"] / (out["pressure_rotation"] + EPS)
    out["axis_x_rotation"] = out["pressure_axis"] * out["rotation"]
    out["rot_pressure_x_rotation"] = out["pressure_rotation"] * out["rotation"]
    out["energy_input_proxy"] = out["pressure_axis"] + out["pressure_rotation"] * out["rotation"]
    out["log_energy_input_proxy"] = np.log1p(out["energy_input_proxy"])

    out["dt"] = out.groupby("well_id")["processing_time"].diff().dt.total_seconds()
    out["dt"] = out["dt"].fillna(out["dt"].median())

    history_cols = [
        "pressure_axis",
        "pressure_rotation",
        "rotation",
        "speed",
        "hardness_score_smooth",
        "energy_input_proxy",
        "pressure_balance",
    ]

    for col in history_cols:
        for lag in [1, 3, 6, 12]:
            out[f"{col}_lag{lag}"] = out.groupby("well_id")[col].shift(lag)

        shifted = out.groupby("well_id")[col].shift(1)
        for w in [6, 12, 30]:
            min_p = max(2, w // 3)
            out[f"{col}_roll_mean_{w}"] = (
                shifted.groupby(out["well_id"])
                       .rolling(w, min_periods=min_p)
                       .mean()
                       .reset_index(level=0, drop=True)
            )
            out[f"{col}_roll_std_{w}"] = (
                shifted.groupby(out["well_id"])
                       .rolling(w, min_periods=min_p)
                       .std()
                       .reset_index(level=0, drop=True)
            )

    for col in ["pressure_axis", "pressure_rotation", "rotation", "speed", "hardness_score_smooth"]:
        prev = out.groupby("well_id")[col].shift(1)
        out[f"{col}_diff1"] = out[col] - prev

    return out


df = add_features(df)
display(df.head())


,processing_time,depth_m,rotation,pressure_axis,pressure_rotation,well_id,speed,dt,total_pressure,pressure_balance,axis_over_rot_pressure,rot_pressure_over_axis,rotation_efficiency,axis_x_rotation,rot_pressure_x_rotation,energy_input_proxy,pseudo_mse,log_energy_input_proxy,log_pseudo_mse,energy_input_proxy_roll_median_12,energy_input_proxy_roll_mean_12,energy_input_proxy_roll_std_12,energy_input_proxy_roll_median_30,energy_input_proxy_roll_mean_30,energy_input_proxy_roll_std_30,energy_input_proxy_roll_median_60,energy_input_proxy_roll_mean_60,energy_input_proxy_roll_std_60,pseudo_mse_roll_median_12,pseudo_mse_roll_mean_12,pseudo_mse_roll_std_12,pseudo_mse_roll_median_30,pseudo_mse_roll_mean_30,pseudo_mse_roll_std_30,pseudo_mse_roll_median_60,pseudo_mse_roll_mean_60,pseudo_mse_roll_std_60,log_pseudo_mse_roll_median_12,log_pseudo_mse_roll_mean_12,log_pseudo_mse_roll_std_12,log_pseudo_mse_roll_median_30,log_pseudo_mse_roll_mean_30,log_pseudo_mse_roll_std_30,log_pseudo_mse_roll_median_60,log_pseudo_mse_roll_mean_60,log_pseudo_mse_roll_std_60,rotation_efficiency_roll_median_12,rotation_efficiency_roll_mean_12,rotation_efficiency_roll_std_12,rotation_efficiency_roll_median_30,rotation_efficiency_roll_mean_30,rotation_efficiency_roll_std_30,rotation_efficiency_roll_median_60,rotation_efficiency_roll_mean_60,rotation_efficiency_roll_std_60,speed_roll_median_12,speed_roll_mean_12,speed_roll_std_12,speed_roll_median_30,speed_roll_mean_30,speed_roll_std_30,speed_roll_median_60,speed_roll_mean_60,speed_roll_std_60,rotation_roll_median_12,rotation_roll_mean_12,rotation_roll_std_12,rotation_roll_median_30,rotation_roll_mean_30,rotation_roll_std_30,rotation_roll_median_60,rotation_roll_mean_60,rotation_roll_std_60,pressure_balance_roll_median_12,pressure_balance_roll_mean_12,pressure_balance_roll_std_12,pressure_balance_roll_median_30,pressure_balance_roll_mean_30,pressure_balance_roll_std_30,pressure_balance_roll_median_60,pressure_balance_roll_mean_60,pressure_balance_roll_std_60,hardness_score_smooth,segment_id,hardness_segment,energy_type_segment_quantile,rock_energy_type_final,pressure_axis_lag1,pressure_axis_lag3,pressure_axis_lag6,pressure_axis_lag12,pressure_axis_roll_mean_6,pressure_axis_roll_std_6,pressure_axis_roll_mean_12,pressure_axis_roll_std_12,pressure_axis_roll_mean_30,pressure_axis_roll_std_30,pressure_rotation_lag1,pressure_rotation_lag3,pressure_rotation_lag6,pressure_rotation_lag12,pressure_rotation_roll_mean_6,pressure_rotation_roll_std_6,pressure_rotation_roll_mean_12,pressure_rotation_roll_std_12,pressure_rotation_roll_mean_30,pressure_rotation_roll_std_30,rotation_lag1,rotation_lag3,rotation_lag6,rotation_lag12,rotation_roll_mean_6,rotation_roll_std_6,speed_lag1,speed_lag3,speed_lag6,speed_lag12,speed_roll_mean_6,speed_roll_std_6,hardness_score_smooth_lag1,hardness_score_smooth_lag3,hardness_score_smooth_lag6,hardness_score_smooth_lag12,hardness_score_smooth_roll_mean_6,hardness_score_smooth_roll_std_6,hardness_score_smooth_roll_mean_12,hardness_score_smooth_roll_std_12,hardness_score_smooth_roll_mean_30,hardness_score_smooth_roll_std_30,energy_input_proxy_lag1,energy_input_proxy_lag3,energy_input_proxy_lag6,energy_input_proxy_lag12,energy_input_proxy_roll_mean_6,energy_input_proxy_roll_std_6,pressure_balance_lag1,pressure_balance_lag3,pressure_balance_lag6,pressure_balance_lag12,pressure_balance_roll_mean_6,pressure_balance_roll_std_6,pressure_axis_diff1,pressure_rotation_diff1,rotation_diff1,speed_diff1,hardness_score_smooth_diff1
0,2025-08-24 10:09:50.980,0.0606,74.256,804,4551,19601,0.002755,5.135,5355,0.150140,0.176664,5.660448,0.016316,59701.824,337939.056,338743.056,1.229314e+08,12.733000,18.627137,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

## 3. Target near_5

In [4]:
def future_mean_by_group(data, value_col, horizon):
    return (
        data.groupby("well_id")[value_col]
            .transform(
                lambda s: (
                    s.shift(-1)
                     .rolling(horizon, min_periods=max(2, horizon // 3))
                     .mean()
                     .shift(-(horizon - 1))
                )
            )
    )

df["target_rotation_near5"] = future_mean_by_group(df, "rotation", TARGET_HORIZON)
df["target_speed_near5"] = future_mean_by_group(df, "speed", TARGET_HORIZON)

# Operator's factual future controls. These are not model targets; they are used only
# for offline sanity checks of recommendation direction.
df["target_pressure_axis_near5"] = future_mean_by_group(df, "pressure_axis", TARGET_HORIZON)
df["target_pressure_rotation_near5"] = future_mean_by_group(df, "pressure_rotation", TARGET_HORIZON)

target_rotation = "target_rotation_near5"
target_speed = "target_speed_near5"
target_pressure_axis = "target_pressure_axis_near5"
target_pressure_rotation = "target_pressure_rotation_near5"

display(
    df[[
        target_rotation,
        target_speed,
        target_pressure_axis,
        target_pressure_rotation,
        "rotation",
        "speed",
        "pressure_axis",
        "pressure_rotation",
    ]].describe(percentiles=[.01, .05, .5, .95, .99])
)


,target_rotation_near5,target_speed_near5,target_pressure_axis_near5,target_pressure_rotation_near5,rotation,speed,pressure_axis,pressure_rotation
count,408225.000000,408225.000000,408225.000000,408225.000000,415049.000000,415049.000000,415049.000000,415049.000000
mean,104.056793,0.013110,17625.738332,14211.245137,103.945315,0.013116,17473.667273,14134.146325
std,10.170965,0.005407,4403.167196,2992.372749,13.370361,0.006615,4682.982816,3243.523440
min,50.868000,0.001409,800.400000,3323.000000,50.010000,0.001002,317.000000,784.000000
1%,68.893104,0.003814,4354.600000,6967.848000,64.980000,0.002755,3713.000000,6271.000000
5%,85.029600,0.005656,7675.120000,8658.000000,81.750000,0.005050,6645.000000,8246.000000
50%,103.308000,0.012524,18859.000000,14751.400000,103.158000,0.012120,18861.000000,14637.000000
95%,123.786000,0.023230,22327.400000,18359.560000,138.474000,0.024240,22343.000000,18758.600000
99%,128.200560,0.028280,23595.752000,20120.656000,139.020000,0.030300,23626.000000,20881.000000
max,139.317600,0.036360,24809.400000,25868.750000,139.578000,0.038957,24872.000000,26318.000000


## 4. Feature lists

Используется pruned feature set. Управляющие признаки `pressure_axis` и `pressure_rotation` оставлены обязательно, потому что optimizer именно ими управляет.

Энергоёмкость подаётся в модель в двух видах:

- `hardness_score_smooth` — устойчивый непрерывный показатель по сглаженному log-pseudo-MSE;
- `rock_energy_type_final` — категориальный уровень энергоёмкости.

По результатам feature-pruning удалены дополнительные multiscale-hardness признаки и relative-diff признаки: они не дали полезного прироста качества.


In [5]:
# Агрессивно сокращённый набор признаков.
# Управляющие признаки p_ax / p_rot оставлены обязательно, даже если их
# permutation importance ниже, потому что optimizer именно ими управляет.
HARDNESS_FEATURE_COLUMNS = [
    "hardness_score_smooth",
]

base_numeric_features = [
    # current controls / state
    "pressure_axis", "pressure_rotation", "pressure_balance",
    "rotation", "speed", "dt",
    "rotation_efficiency", "axis_x_rotation",
    "energy_input_proxy",

    # stable energy state
    *HARDNESS_FEATURE_COLUMNS,

    # pressure history: оставляем pressure_axis history,
    # pressure_rotation lag-и удалены как слабые/дублирующие
    "pressure_axis_lag1", "pressure_axis_lag3", "pressure_axis_lag6",

    # rotation history
    "rotation_lag1", "rotation_lag3", "rotation_lag6",

    # speed history: lag6 удалён как слабый кандидат
    "speed_lag1", "speed_lag3",

    # stable hardness history
    "hardness_score_smooth_lag1", "hardness_score_smooth_lag3", "hardness_score_smooth_lag6",

    # rolling context
    "pressure_axis_roll_mean_12", "pressure_axis_roll_std_12",
    "pressure_rotation_roll_mean_12", "pressure_rotation_roll_std_12",
    "rotation_roll_mean_12", "rotation_roll_std_12",
    "speed_roll_mean_12", "speed_roll_std_12",
    "hardness_score_smooth_roll_mean_12",

    # short dynamics: relative diffs removed after feature-pruning
    "pressure_axis_diff1", "pressure_rotation_diff1", "rotation_diff1",
    "speed_diff1",
]

categorical_features = ["rock_energy_type_final"]

speed_extra_features = ["candidate_target_rotation"]
speed_numeric_features = base_numeric_features + speed_extra_features

PRUNED_FEATURES_REMOVED = [
    # multiscale hardness removed after feature-pruning
    "hardness_score",
    "hardness_score_smooth_12",
    "hardness_score_smooth_30",

    # relative diffs removed after feature-pruning
    "pressure_axis_rel_diff1",
    "pressure_rotation_rel_diff1",
    "rotation_rel_diff1",
    "speed_rel_diff1",
    "hardness_score_smooth_rel_diff1",

    # previous aggressive-pruning removals
    "axis_over_rot_pressure",
    "rot_pressure_over_axis",
    "log_energy_input_proxy",
    "rot_pressure_x_rotation",
    "total_pressure",
    "speed_lag6",
    "pressure_rotation_lag1",
    "pressure_rotation_lag3",
    "pressure_rotation_lag6",
    "hardness_score_smooth_roll_std_12",
    "hardness_score_smooth_diff1",
]

all_required = base_numeric_features + categorical_features + [target_rotation, target_speed]
missing = [c for c in all_required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

model_df = df.dropna(subset=all_required + ["well_id", "processing_time"]).copy()

print("Model df:", model_df.shape)
print("Numeric features:", len(base_numeric_features))
print("Categorical features:", categorical_features)
print("Hardness features:", HARDNESS_FEATURE_COLUMNS)
print("Removed features:", PRUNED_FEATURES_REMOVED)


Model df: (365575, 150)
Numeric features: 34
Categorical features: ['rock_energy_type_final']
Hardness features: ['hardness_score_smooth']
Removed features: ['hardness_score', 'hardness_score_smooth_12', 'hardness_score_smooth_30', 'pressure_axis_rel_diff1', 'pressure_rotation_rel_diff1', 'rotation_rel_diff1', 'speed_rel_diff1', 'hardness_score_smooth_rel_diff1', 'axis_over_rot_pressure', 'rot_pressure_over_axis', 'log_energy_input_proxy', 'rot_pressure_x_rotation', 'total_pressure', 'speed_lag6', 'pressure_rotation_lag1', 'pressure_rotation_lag3', 'pressure_rotation_lag6', 'hardness_score_smooth_roll_std_12', 'hardness_score_smooth_diff1']




5. Split by unseen wells
¶



In [6]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(model_df, groups=model_df["well_id"]))

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("Train:", train_df.shape, "wells:", train_df["well_id"].nunique())
print("Test:", test_df.shape, "wells:", test_df["well_id"].nunique())
print("Test wells:", list(test_df["well_id"].unique())[:10])


Train: (271790, 150) wells: 1279
Test: (93785, 150) wells: 427
Test wells: [np.int64(19677), np.int64(19720), np.int64(19775), np.int64(19780), np.int64(19789), np.int64(19905), np.int64(19909), np.int64(19935), np.int64(19958), np.int64(19977)]




6. Train LightGBM models
¶



In [7]:
def make_regressor():
    return lgb.LGBMRegressor(
        n_estimators=650,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        objective="regression",
        verbosity=-1,
    )


def make_preprocessor(numeric_features, categorical_features):
    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            ("num", "passthrough", numeric_features),
        ],
        remainder="drop",
    )


def regression_metrics(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(root_mean_squared_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
    }


rotation_model = Pipeline(steps=[
    ("preprocess", make_preprocessor(base_numeric_features, categorical_features)),
    ("model", make_regressor()),
])

rotation_model.fit(
    train_df[base_numeric_features + categorical_features],
    train_df[target_rotation],
)

test_df["pred_target_rotation"] = rotation_model.predict(
    test_df[base_numeric_features + categorical_features]
)

rotation_metrics = regression_metrics(
    test_df[target_rotation],
    test_df["pred_target_rotation"],
)

print("Rotation model near_5:")
print(rotation_metrics)


train_speed_df = train_df.copy()
train_speed_df["candidate_target_rotation"] = train_speed_df[target_rotation]

test_speed_oracle_df = test_df.copy()
test_speed_oracle_df["candidate_target_rotation"] = test_speed_oracle_df[target_rotation]

test_speed_chained_df = test_df.copy()
test_speed_chained_df["candidate_target_rotation"] = test_speed_chained_df["pred_target_rotation"]

speed_model = Pipeline(steps=[
    ("preprocess", make_preprocessor(speed_numeric_features, categorical_features)),
    ("model", make_regressor()),
])

speed_model.fit(
    train_speed_df[speed_numeric_features + categorical_features],
    train_speed_df[target_speed],
)

test_df["pred_target_speed_oracle_rotation"] = speed_model.predict(
    test_speed_oracle_df[speed_numeric_features + categorical_features]
)

test_df["pred_target_speed_chained"] = speed_model.predict(
    test_speed_chained_df[speed_numeric_features + categorical_features]
)

speed_metrics_oracle = regression_metrics(
    test_df[target_speed],
    test_df["pred_target_speed_oracle_rotation"],
)

speed_metrics_chained = regression_metrics(
    test_df[target_speed],
    test_df["pred_target_speed_chained"],
)

print("Speed model near_5 with actual target rotation:")
print(speed_metrics_oracle)

print("Speed model near_5 chained:")
print(speed_metrics_chained)


Rotation model near_5:
{'MAE': 1.7420703857850104, 'RMSE': 3.5930753626617626, 'R2': 0.8172769637719016}
Speed model near_5 with actual target rotation:
{'MAE': 0.0019392085621642466, 'RMSE': 0.002626100319487699, 'R2': 0.7541302757810525}
Speed model near_5 chained:
{'MAE': 0.002033990595436879, 'RMSE': 0.002738528832262074, 'R2': 0.7326272983014825}


## 7. Baselines and surface ranges

In [8]:
test_df["baseline_current_speed"] = test_df["speed"]
test_df["baseline_speed_roll_mean_12"] = test_df["speed_roll_mean_12"].fillna(test_df["speed"])

baseline_compare = pd.DataFrame([
    {"model": "current_speed", **regression_metrics(test_df[target_speed], test_df["baseline_current_speed"])},
    {"model": "speed_roll_mean_12", **regression_metrics(test_df[target_speed], test_df["baseline_speed_roll_mean_12"])},
    {"model": "rotation_to_speed_chained_near5", **speed_metrics_chained},
    {"model": "speed_oracle_rotation_near5", **speed_metrics_oracle},
]).sort_values("MAE")

display(baseline_compare)

surface_ranges = {}

for et, part in train_df.groupby("rock_energy_type_final"):
    if len(part) < 100:
        continue

    surface_ranges[et] = {
        "pressure_axis_q05": float(part["pressure_axis"].quantile(0.05)),
        "pressure_axis_q95": float(part["pressure_axis"].quantile(0.95)),
        "pressure_rotation_q05": float(part["pressure_rotation"].quantile(0.05)),
        "pressure_rotation_q95": float(part["pressure_rotation"].quantile(0.95)),
        "rotation_median": float(part["rotation"].median()),
        "speed_median": float(part["speed"].median()),
        "hardness_median": float(part["hardness_score_smooth"].median()),
        "rows": int(len(part)),
    }

surface_ranges_df = pd.DataFrame(surface_ranges).T
display(surface_ranges_df)


,model,MAE,RMSE,R2
3,speed_oracle_rotation_near5,0.001939,0.002626,0.754130
2,rotation_to_speed_chained_near5,0.002034,0.002739,0.732627
1,speed_roll_mean_12,0.002483,0.003403,0.587099
0,current_speed,0.003512,0.004528,0.268915


,pressure_axis_q05,pressure_axis_q95,pressure_rotation_q05,pressure_rotation_q95,rotation_median,speed_median,hardness_median,rows
hard_high_energy,13522.0,22891.0,9896.55,18835.00,103.458,0.00606,1.109717,67352.0
medium_high_energy,14055.1,22685.0,10468.00,18882.00,102.864,0.01212,0.311667,69363.0
medium_low_energy,12547.0,22352.8,9755.20,19057.00,102.966,0.01212,-0.213656,68165.0
soft_low_energy,7527.0,21376.0,8315.00,18579.55,103.410,0.01818,-1.071560,66910.0


## 8. Final light-penalty optimizer

In [9]:
def recompute_candidate_features(grid):
    grid = grid.copy()
    grid["total_pressure"] = grid["pressure_axis"] + grid["pressure_rotation"]
    grid["pressure_balance"] = grid["pressure_axis"] / (grid["total_pressure"] + EPS)
    grid["rotation_efficiency"] = grid["rotation"] / (grid["pressure_rotation"] + EPS)
    grid["axis_x_rotation"] = grid["pressure_axis"] * grid["rotation"]
    grid["energy_input_proxy"] = grid["pressure_axis"] + grid["pressure_rotation"] * grid["rotation"]
    return grid


def build_candidate_grid(row, grid_size=GRID_SIZE, max_delta_frac=MAX_DELTA_FRAC):
    et = row["rock_energy_type_final"]

    if et in surface_ranges:
        r = surface_ranges[et]
        p_ax_low = r["pressure_axis_q05"]
        p_ax_high = r["pressure_axis_q95"]
        p_rot_low = r["pressure_rotation_q05"]
        p_rot_high = r["pressure_rotation_q95"]
    else:
        p_ax_low = train_df["pressure_axis"].quantile(0.05)
        p_ax_high = train_df["pressure_axis"].quantile(0.95)
        p_rot_low = train_df["pressure_rotation"].quantile(0.05)
        p_rot_high = train_df["pressure_rotation"].quantile(0.95)

    cur_ax = row["pressure_axis"]
    cur_rotp = row["pressure_rotation"]

    local_ax_low = cur_ax * (1.0 - max_delta_frac)
    local_ax_high = cur_ax * (1.0 + max_delta_frac)
    local_rotp_low = cur_rotp * (1.0 - max_delta_frac)
    local_rotp_high = cur_rotp * (1.0 + max_delta_frac)

    p_ax_min = max(p_ax_low, local_ax_low)
    p_ax_max = min(p_ax_high, local_ax_high)
    p_rot_min = max(p_rot_low, local_rotp_low)
    p_rot_max = min(p_rot_high, local_rotp_high)

    if p_ax_min >= p_ax_max:
        p_ax_min, p_ax_max = local_ax_low, local_ax_high

    if p_rot_min >= p_rot_max:
        p_rot_min, p_rot_max = local_rotp_low, local_rotp_high

    p_ax_grid = np.linspace(p_ax_min, p_ax_max, grid_size)
    p_rot_grid = np.linspace(p_rot_min, p_rot_max, grid_size)
    PA, PR = np.meshgrid(p_ax_grid, p_rot_grid)

    grid = pd.DataFrame({"pressure_axis": PA.ravel(), "pressure_rotation": PR.ravel()})

    recomputed = {
        "pressure_axis", "pressure_rotation", "pressure_balance",
        "rotation_efficiency", "axis_x_rotation", "energy_input_proxy",
    }

    for col in base_numeric_features:
        if col not in recomputed:
            grid[col] = row[col]

    for col in categorical_features:
        grid[col] = row[col]

    grid = recompute_candidate_features(grid)

    grid["delta_pressure_axis_frac"] = grid["pressure_axis"] / (row["pressure_axis"] + EPS) - 1.0
    grid["delta_pressure_rotation_frac"] = grid["pressure_rotation"] / (row["pressure_rotation"] + EPS) - 1.0

    return grid, PA, PR


def predict_current_target_speed(row):
    current_grid = pd.DataFrame([row[base_numeric_features + categorical_features].to_dict()])
    current_grid["candidate_target_rotation"] = rotation_model.predict(
        current_grid[base_numeric_features + categorical_features]
    )
    return float(speed_model.predict(current_grid[speed_numeric_features + categorical_features])[0])


def add_light_penalized_score(grid, current_pred_speed):
    grid = grid.copy()

    speed_scale = max(abs(current_pred_speed), EPS)

    axis_delta_norm = np.abs(grid["delta_pressure_axis_frac"]) / MAX_DELTA_FRAC
    rot_delta_norm = np.abs(grid["delta_pressure_rotation_frac"]) / MAX_DELTA_FRAC

    change_penalty = (
        CHANGE_PENALTY_WEIGHT
        * speed_scale
        * (axis_delta_norm**2 + rot_delta_norm**2)
        / 2.0
    )

    axis_edge = np.clip((axis_delta_norm - BOUNDARY_START) / (1.0 - BOUNDARY_START + EPS), 0, 1)
    rot_edge = np.clip((rot_delta_norm - BOUNDARY_START) / (1.0 - BOUNDARY_START + EPS), 0, 1)

    boundary_penalty = (
        BOUNDARY_PENALTY_WEIGHT
        * speed_scale
        * (axis_edge**2 + rot_edge**2)
        / 2.0
    )

    grid["change_penalty"] = change_penalty
    grid["boundary_penalty"] = boundary_penalty
    grid["optimizer_score"] = grid["pred_target_speed"] - change_penalty - boundary_penalty

    return grid


def recommend_for_row(row, grid_size=GRID_SIZE):
    grid, PA, PR = build_candidate_grid(row, grid_size=grid_size)

    grid["candidate_target_rotation"] = rotation_model.predict(
        grid[base_numeric_features + categorical_features]
    )

    grid["pred_target_speed"] = speed_model.predict(
        grid[speed_numeric_features + categorical_features]
    )

    current_pred_speed = predict_current_target_speed(row)

    grid = add_light_penalized_score(grid, current_pred_speed=current_pred_speed)

    best_idx = int(grid["optimizer_score"].values.argmax())
    best = grid.iloc[best_idx].copy()

    return {
        "recommended_pressure_axis": float(best["pressure_axis"]),
        "recommended_pressure_rotation": float(best["pressure_rotation"]),
        "predicted_target_rotation": float(best["candidate_target_rotation"]),
        "predicted_target_speed": float(best["pred_target_speed"]),
        "optimizer_score": float(best["optimizer_score"]),
        "change_penalty": float(best["change_penalty"]),
        "boundary_penalty": float(best["boundary_penalty"]),
        "current_predicted_target_speed": current_pred_speed,
        "predicted_uplift_pct": 100.0 * (float(best["pred_target_speed"]) / (current_pred_speed + EPS) - 1.0),
        "score_uplift_pct": 100.0 * (float(best["optimizer_score"]) / (current_pred_speed + EPS) - 1.0),
        "delta_pressure_axis_pct": 100.0 * (float(best["pressure_axis"]) / (row["pressure_axis"] + EPS) - 1.0),
        "delta_pressure_rotation_pct": 100.0 * (float(best["pressure_rotation"]) / (row["pressure_rotation"] + EPS) - 1.0),
        "grid": grid,
        "PA": PA,
        "PR": PR,
        "Z": grid["pred_target_speed"].values.reshape(PA.shape),
        "Z_score": grid["optimizer_score"].values.reshape(PA.shape),
    }


## Offline recommendation replay with operator-comparison metrics

В этом блоке считаются несколько offline-оценок качества рекомендации.

1. `predicted_uplift_pct` — основной model-based uplift: прогноз speed для рекомендованного кандидата относительно прогноза speed при текущем режиме оператора.
2. `predicted_regret_vs_operator_pct` — прогнозируемый regret относительно фактического результата оператора: прогноз speed для рекомендованного кандидата относительно реальной средней скорости оператора на следующих 5 шагах.
3. `recommended_win_vs_operator_*` — доля строк, где прогноз рекомендации выше фактического результата оператора с запасом 0%, 2% или 5%.
4. `current_prediction_error_vs_operator_pct` — диагностическая ошибка прогноза текущего режима относительно фактического будущего оператора.
5. `bias_adjusted_regret_vs_operator_pct` — regret относительно оператора с поправкой на bias прогноза текущего режима.
6. `same_direction_*` — sanity-check направления управления: совпадает ли направление изменения давления, рекомендованное моделью, с тем, куда оператор реально двигался в следующие 5 шагов.

Все operator-comparison метрики являются offline counterfactual diagnostics. Они не являются фактически измеренным causal uplift, потому что в данных оператор не применял рекомендованный режим.


In [10]:
EVAL_N = min(2500, len(test_df))
eval_points = test_df.sample(EVAL_N, random_state=RANDOM_STATE).copy()

DIRECTION_DEADBAND_PCT = 0.5


def signed_direction(delta_pct: float, deadband_pct: float = DIRECTION_DEADBAND_PCT) -> int:
    if delta_pct > deadband_pct:
        return 1
    if delta_pct < -deadband_pct:
        return -1
    return 0


def safe_mean(series: pd.Series) -> float:
    if len(series) == 0:
        return np.nan
    return float(series.mean())


recommendations = []

for _, row in eval_points.iterrows():
    rec = recommend_for_row(row, grid_size=GRID_SIZE)

    actual_operator_target_speed = float(row[target_speed])
    current_predicted_target_speed = float(rec["current_predicted_target_speed"])
    recommended_predicted_target_speed = float(rec["predicted_target_speed"])

    # Main model-based uplift:
    # recommended prediction vs current-mode prediction.
    predicted_uplift_pct = 100.0 * (
        recommended_predicted_target_speed / (current_predicted_target_speed + EPS) - 1.0
    )

    # Operator-comparison regret:
    # recommended prediction vs actual operator outcome on the same near_5 horizon.
    predicted_regret_vs_operator_speed = recommended_predicted_target_speed - actual_operator_target_speed
    predicted_regret_vs_operator_pct = 100.0 * (
        recommended_predicted_target_speed / (actual_operator_target_speed + EPS) - 1.0
    )

    # Diagnostic: how much the current-mode model prediction differs from actual operator outcome.
    current_prediction_error_vs_operator_speed = current_predicted_target_speed - actual_operator_target_speed
    current_prediction_error_vs_operator_pct = 100.0 * (
        current_predicted_target_speed / (actual_operator_target_speed + EPS) - 1.0
    )

    # Bias-adjusted regret: recommendation-vs-operator minus current-prediction bias.
    bias_adjusted_regret_vs_operator_pct = (
        predicted_regret_vs_operator_pct - current_prediction_error_vs_operator_pct
    )

    operator_future_pressure_axis = float(row.get(target_pressure_axis, np.nan))
    operator_future_pressure_rotation = float(row.get(target_pressure_rotation, np.nan))

    operator_delta_pressure_axis_pct = 100.0 * (
        operator_future_pressure_axis / (float(row["pressure_axis"]) + EPS) - 1.0
    )
    operator_delta_pressure_rotation_pct = 100.0 * (
        operator_future_pressure_rotation / (float(row["pressure_rotation"]) + EPS) - 1.0
    )

    rec_delta_axis_pct = float(rec["delta_pressure_axis_pct"])
    rec_delta_rot_pct = float(rec["delta_pressure_rotation_pct"])

    rec_axis_direction = signed_direction(rec_delta_axis_pct)
    rec_rot_direction = signed_direction(rec_delta_rot_pct)
    operator_axis_direction = signed_direction(operator_delta_pressure_axis_pct)
    operator_rot_direction = signed_direction(operator_delta_pressure_rotation_pct)

    recommendations.append({
        "well_id": row["well_id"],
        "processing_time": row["processing_time"],
        "rock_energy_type_final": row["rock_energy_type_final"],
        "operator_pressure_axis": row["pressure_axis"],
        "operator_pressure_rotation": row["pressure_rotation"],
        "operator_future_pressure_axis_near5": operator_future_pressure_axis,
        "operator_future_pressure_rotation_near5": operator_future_pressure_rotation,
        "recommended_pressure_axis": rec["recommended_pressure_axis"],
        "recommended_pressure_rotation": rec["recommended_pressure_rotation"],
        "current_speed": row["speed"],
        "target_speed_actual": actual_operator_target_speed,
        "current_predicted_target_speed": current_predicted_target_speed,
        "recommended_predicted_target_speed": recommended_predicted_target_speed,
        "predicted_regret_vs_operator_speed": predicted_regret_vs_operator_speed,
        "current_prediction_error_vs_operator_speed": current_prediction_error_vs_operator_speed,
        "optimizer_score": rec["optimizer_score"],
        "change_penalty": rec["change_penalty"],
        "boundary_penalty": rec["boundary_penalty"],
        "predicted_uplift_pct": predicted_uplift_pct,
        "score_uplift_pct": rec["score_uplift_pct"],
        "predicted_regret_vs_operator_pct": predicted_regret_vs_operator_pct,
        # Backward-compatible alias from the previous notebook version.
        "predicted_uplift_vs_actual_pct": predicted_regret_vs_operator_pct,
        "current_prediction_error_vs_operator_pct": current_prediction_error_vs_operator_pct,
        # Backward-compatible alias from the previous notebook version.
        "current_prediction_error_vs_actual_pct": current_prediction_error_vs_operator_pct,
        "bias_adjusted_regret_vs_operator_pct": bias_adjusted_regret_vs_operator_pct,
        "delta_pressure_axis_pct": rec_delta_axis_pct,
        "delta_pressure_rotation_pct": rec_delta_rot_pct,
        "operator_delta_pressure_axis_pct": operator_delta_pressure_axis_pct,
        "operator_delta_pressure_rotation_pct": operator_delta_pressure_rotation_pct,
        "rec_axis_direction": rec_axis_direction,
        "rec_rot_direction": rec_rot_direction,
        "operator_axis_direction": operator_axis_direction,
        "operator_rot_direction": operator_rot_direction,
    })

rec_df = pd.DataFrame(recommendations)

boundary_tol = 0.95 * MAX_DELTA_FRAC * 100.0
rec_df["axis_near_boundary"] = rec_df["delta_pressure_axis_pct"].abs() >= boundary_tol
rec_df["rot_near_boundary"] = rec_df["delta_pressure_rotation_pct"].abs() >= boundary_tol
rec_df["any_boundary"] = rec_df["axis_near_boundary"] | rec_df["rot_near_boundary"]

# Recommended prediction vs actual operator outcome.
rec_df["recommended_win_vs_operator"] = (
    rec_df["recommended_predicted_target_speed"] > rec_df["target_speed_actual"]
)
rec_df["recommended_win_vs_operator_2pct"] = (
    rec_df["recommended_predicted_target_speed"] > rec_df["target_speed_actual"] * 1.02
)
rec_df["recommended_win_vs_operator_5pct"] = (
    rec_df["recommended_predicted_target_speed"] > rec_df["target_speed_actual"] * 1.05
)
rec_df["current_pred_above_operator"] = (
    rec_df["current_predicted_target_speed"] > rec_df["target_speed_actual"]
)
# Backward-compatible aliases.
rec_df["recommended_pred_above_actual"] = rec_df["recommended_win_vs_operator"]
rec_df["current_pred_above_actual"] = rec_df["current_pred_above_operator"]

# Direction sanity checks.
rec_df["same_direction_axis"] = rec_df["rec_axis_direction"] == rec_df["operator_axis_direction"]
rec_df["same_direction_rotation"] = rec_df["rec_rot_direction"] == rec_df["operator_rot_direction"]
rec_df["same_direction_both"] = rec_df["same_direction_axis"] & rec_df["same_direction_rotation"]

rec_df["direction_comparable_axis"] = (
    (rec_df["rec_axis_direction"] != 0) | (rec_df["operator_axis_direction"] != 0)
)
rec_df["direction_comparable_rotation"] = (
    (rec_df["rec_rot_direction"] != 0) | (rec_df["operator_rot_direction"] != 0)
)
rec_df["direction_comparable_both"] = (
    rec_df["direction_comparable_axis"] | rec_df["direction_comparable_rotation"]
)

optimizer_summary = pd.DataFrame([{
    "optimizer_mode": FINAL_OPTIMIZER_MODE,
    "rows": len(rec_df),

    # Main model-based uplift: recommended prediction vs current prediction.
    "median_uplift_pct": rec_df["predicted_uplift_pct"].median(),
    "mean_uplift_pct": rec_df["predicted_uplift_pct"].mean(),
    "p05_uplift_pct": rec_df["predicted_uplift_pct"].quantile(0.05),
    "p95_uplift_pct": rec_df["predicted_uplift_pct"].quantile(0.95),
    "median_score_uplift_pct": rec_df["score_uplift_pct"].median(),

    # Operator-comparison regret: recommended prediction vs actual operator near_5 outcome.
    "median_predicted_regret_vs_operator_pct": rec_df["predicted_regret_vs_operator_pct"].median(),
    "mean_predicted_regret_vs_operator_pct": rec_df["predicted_regret_vs_operator_pct"].mean(),
    "p05_predicted_regret_vs_operator_pct": rec_df["predicted_regret_vs_operator_pct"].quantile(0.05),
    "p25_predicted_regret_vs_operator_pct": rec_df["predicted_regret_vs_operator_pct"].quantile(0.25),
    "p75_predicted_regret_vs_operator_pct": rec_df["predicted_regret_vs_operator_pct"].quantile(0.75),
    "p95_predicted_regret_vs_operator_pct": rec_df["predicted_regret_vs_operator_pct"].quantile(0.95),
    "recommended_win_vs_operator_rate": rec_df["recommended_win_vs_operator"].mean(),
    "recommended_win_vs_operator_2pct_rate": rec_df["recommended_win_vs_operator_2pct"].mean(),
    "recommended_win_vs_operator_5pct_rate": rec_df["recommended_win_vs_operator_5pct"].mean(),

    # Backward-compatible names from the previous notebook version.
    "median_uplift_vs_actual_pct": rec_df["predicted_regret_vs_operator_pct"].median(),
    "mean_uplift_vs_actual_pct": rec_df["predicted_regret_vs_operator_pct"].mean(),
    "share_recommended_pred_above_actual": rec_df["recommended_win_vs_operator"].mean(),

    # Diagnostic: current-mode prediction error vs actual operator near_5 outcome.
    "median_current_prediction_error_vs_operator_pct": rec_df["current_prediction_error_vs_operator_pct"].median(),
    "mean_current_prediction_error_vs_operator_pct": rec_df["current_prediction_error_vs_operator_pct"].mean(),
    "p05_current_prediction_error_vs_operator_pct": rec_df["current_prediction_error_vs_operator_pct"].quantile(0.05),
    "p95_current_prediction_error_vs_operator_pct": rec_df["current_prediction_error_vs_operator_pct"].quantile(0.95),
    "share_current_pred_above_operator": rec_df["current_pred_above_operator"].mean(),

    # Bias-adjusted regret diagnostic.
    "median_bias_adjusted_regret_vs_operator_pct": rec_df["bias_adjusted_regret_vs_operator_pct"].median(),
    "mean_bias_adjusted_regret_vs_operator_pct": rec_df["bias_adjusted_regret_vs_operator_pct"].mean(),

    # Direction sanity checks.
    "same_direction_axis_rate": safe_mean(
        rec_df.loc[rec_df["direction_comparable_axis"], "same_direction_axis"]
    ),
    "same_direction_rotation_rate": safe_mean(
        rec_df.loc[rec_df["direction_comparable_rotation"], "same_direction_rotation"]
    ),
    "same_direction_both_rate": safe_mean(
        rec_df.loc[rec_df["direction_comparable_both"], "same_direction_both"]
    ),

    "median_delta_axis_pct": rec_df["delta_pressure_axis_pct"].median(),
    "median_delta_rot_pct": rec_df["delta_pressure_rotation_pct"].median(),
    "median_abs_delta_axis_pct": rec_df["delta_pressure_axis_pct"].abs().median(),
    "median_abs_delta_rot_pct": rec_df["delta_pressure_rotation_pct"].abs().median(),
    "median_operator_delta_axis_pct": rec_df["operator_delta_pressure_axis_pct"].median(),
    "median_operator_delta_rot_pct": rec_df["operator_delta_pressure_rotation_pct"].median(),
    "axis_boundary_ratio": rec_df["axis_near_boundary"].mean(),
    "rot_boundary_ratio": rec_df["rot_near_boundary"].mean(),
    "any_boundary_ratio": rec_df["any_boundary"].mean(),
}])

display(optimizer_summary)

by_energy = (
    rec_df
    .groupby("rock_energy_type_final")
    .agg(
        rows=("predicted_uplift_pct", "size"),
        median_uplift_pct=("predicted_uplift_pct", "median"),
        mean_uplift_pct=("predicted_uplift_pct", "mean"),
        p05_uplift_pct=("predicted_uplift_pct", lambda s: s.quantile(0.05)),
        p95_uplift_pct=("predicted_uplift_pct", lambda s: s.quantile(0.95)),
        median_predicted_regret_vs_operator_pct=("predicted_regret_vs_operator_pct", "median"),
        mean_predicted_regret_vs_operator_pct=("predicted_regret_vs_operator_pct", "mean"),
        p05_predicted_regret_vs_operator_pct=("predicted_regret_vs_operator_pct", lambda s: s.quantile(0.05)),
        p95_predicted_regret_vs_operator_pct=("predicted_regret_vs_operator_pct", lambda s: s.quantile(0.95)),
        recommended_win_vs_operator_rate=("recommended_win_vs_operator", "mean"),
        recommended_win_vs_operator_2pct_rate=("recommended_win_vs_operator_2pct", "mean"),
        recommended_win_vs_operator_5pct_rate=("recommended_win_vs_operator_5pct", "mean"),
        median_current_prediction_error_vs_operator_pct=("current_prediction_error_vs_operator_pct", "median"),
        mean_current_prediction_error_vs_operator_pct=("current_prediction_error_vs_operator_pct", "mean"),
        median_bias_adjusted_regret_vs_operator_pct=("bias_adjusted_regret_vs_operator_pct", "median"),
        share_current_pred_above_operator=("current_pred_above_operator", "mean"),
        median_delta_axis_pct=("delta_pressure_axis_pct", "median"),
        median_delta_rot_pct=("delta_pressure_rotation_pct", "median"),
        median_abs_delta_axis_pct=("delta_pressure_axis_pct", lambda s: s.abs().median()),
        median_abs_delta_rot_pct=("delta_pressure_rotation_pct", lambda s: s.abs().median()),
        median_operator_delta_axis_pct=("operator_delta_pressure_axis_pct", "median"),
        median_operator_delta_rot_pct=("operator_delta_pressure_rotation_pct", "median"),
        boundary_ratio=("any_boundary", "mean"),
    )
    .reset_index()
)

display(by_energy)

operator_comparison_summary = pd.DataFrame([
    {
        "metric": "model_based_uplift_pct = recommended_pred / current_pred - 1",
        "median": rec_df["predicted_uplift_pct"].median(),
        "mean": rec_df["predicted_uplift_pct"].mean(),
        "p05": rec_df["predicted_uplift_pct"].quantile(0.05),
        "p25": rec_df["predicted_uplift_pct"].quantile(0.25),
        "p75": rec_df["predicted_uplift_pct"].quantile(0.75),
        "p95": rec_df["predicted_uplift_pct"].quantile(0.95),
    },
    {
        "metric": "predicted_regret_vs_operator_pct = recommended_pred / actual_operator_near5 - 1",
        "median": rec_df["predicted_regret_vs_operator_pct"].median(),
        "mean": rec_df["predicted_regret_vs_operator_pct"].mean(),
        "p05": rec_df["predicted_regret_vs_operator_pct"].quantile(0.05),
        "p25": rec_df["predicted_regret_vs_operator_pct"].quantile(0.25),
        "p75": rec_df["predicted_regret_vs_operator_pct"].quantile(0.75),
        "p95": rec_df["predicted_regret_vs_operator_pct"].quantile(0.95),
    },
    {
        "metric": "current_prediction_error_vs_operator_pct = current_pred / actual_operator_near5 - 1",
        "median": rec_df["current_prediction_error_vs_operator_pct"].median(),
        "mean": rec_df["current_prediction_error_vs_operator_pct"].mean(),
        "p05": rec_df["current_prediction_error_vs_operator_pct"].quantile(0.05),
        "p25": rec_df["current_prediction_error_vs_operator_pct"].quantile(0.25),
        "p75": rec_df["current_prediction_error_vs_operator_pct"].quantile(0.75),
        "p95": rec_df["current_prediction_error_vs_operator_pct"].quantile(0.95),
    },
    {
        "metric": "bias_adjusted_regret_vs_operator_pct = regret_vs_operator - current_prediction_error",
        "median": rec_df["bias_adjusted_regret_vs_operator_pct"].median(),
        "mean": rec_df["bias_adjusted_regret_vs_operator_pct"].mean(),
        "p05": rec_df["bias_adjusted_regret_vs_operator_pct"].quantile(0.05),
        "p25": rec_df["bias_adjusted_regret_vs_operator_pct"].quantile(0.25),
        "p75": rec_df["bias_adjusted_regret_vs_operator_pct"].quantile(0.75),
        "p95": rec_df["bias_adjusted_regret_vs_operator_pct"].quantile(0.95),
    },
])

display(operator_comparison_summary)

win_rate_summary = pd.DataFrame([
    {"metric": "recommended_win_vs_operator_rate", "value": rec_df["recommended_win_vs_operator"].mean()},
    {"metric": "recommended_win_vs_operator_2pct_rate", "value": rec_df["recommended_win_vs_operator_2pct"].mean()},
    {"metric": "recommended_win_vs_operator_5pct_rate", "value": rec_df["recommended_win_vs_operator_5pct"].mean()},
    {"metric": "same_direction_axis_rate", "value": optimizer_summary.loc[0, "same_direction_axis_rate"]},
    {"metric": "same_direction_rotation_rate", "value": optimizer_summary.loc[0, "same_direction_rotation_rate"]},
    {"metric": "same_direction_both_rate", "value": optimizer_summary.loc[0, "same_direction_both_rate"]},
])

display(win_rate_summary)

# Backward-compatible name used by the previous notebook version.
uplift_compare = operator_comparison_summary.copy()


,optimizer_mode,rows,median_uplift_pct,mean_uplift_pct,p05_uplift_pct,p95_uplift_pct,median_score_uplift_pct,median_predicted_regret_vs_operator_pct,mean_predicted_regret_vs_operator_pct,p05_predicted_regret_vs_operator_pct,p25_predicted_regret_vs_operator_pct,p75_predicted_regret_vs_operator_pct,p95_predicted_regret_vs_operator_pct,recommended_win_vs_operator_rate,recommended_win_vs_operator_2pct_rate,recommended_win_vs_operator_5pct_rate,median_uplift_vs_actual_pct,mean_uplift_vs_actual_pct,share_recommended_pred_above_actual,median_current_prediction_error_vs_operator_pct,mean_current_prediction_error_vs_operator_pct,p05_current_prediction_error_vs_operator_pct,p95_current_prediction_error_vs_operator_pct,share_current_pred_above_operator,median_bias_adjusted_regret_vs_operator_pct,mean_bias_adjusted_regret_vs_operator_pct,same_direction_axis_rate,same_direction_rotation_rate,same_direction_both_rate,median_delta_axis_pct,median_delta_rot_pct,median_abs_delta_axis_pct,median_abs_delta_rot_pct,median_operator_delta_axis_pct,median_operator_delta_rot_pct,axis_boundary_ratio,rot_boundary_ratio,any_boundary_ratio
0,light_penalty,2500,2.244168,2.938223,0.146377,8.118032,1.833432,4.670562,7.930803,-24.284943,-7.6123,19.771873,50.227729,0.6012,0.5584,0.4912,4.670562,7.930803,0.6012,1.695947,4.850874,-26.695159,45.46443,0.54,2.286845,3.079929,0.09095,0.489583,0.0988,-1.291744e-11,5.6,3.2,5.6,0.033511,0.690659,0.0084,0.0336,0.0404


,rock_energy_type_final,rows,median_uplift_pct,mean_uplift_pct,p05_uplift_pct,p95_uplift_pct,median_predicted_regret_vs_operator_pct,mean_predicted_regret_vs_operator_pct,p05_predicted_regret_vs_operator_pct,p95_predicted_regret_vs_operator_pct,recommended_win_vs_operator_rate,recommended_win_vs_operator_2pct_rate,recommended_win_vs_operator_5pct_rate,median_current_prediction_error_vs_operator_pct,mean_current_prediction_error_vs_operator_pct,median_bias_adjusted_regret_vs_operator_pct,share_current_pred_above_operator,median_delta_axis_pct,median_delta_rot_pct,median_abs_delta_axis_pct,median_abs_delta_rot_pct,median_operator_delta_axis_pct,median_operator_delta_rot_pct,boundary_ratio
0,hard_high_energy,655,4.098554,4.766702,0.820450,10.835769,8.233789,11.842244,-23.729607,60.144318,0.654962,0.618321,0.570992,3.634594,6.769386,4.141265,0.554198,-8.000000e-01,6.4,4.000000,6.400000,0.026888,0.862948,0.070229
1,medium_high_energy,584,2.768525,3.252062,0.482584,7.804981,5.439126,8.300654,-22.769739,44.933052,0.633562,0.585616,0.506849,2.326806,4.923690,2.811830,0.566781,-8.422327e-01,6.4,2.559985,6.400000,0.019618,-0.137296,0.046233
2,medium_low_energy,661,1.611927,2.137216,0.133937,5.853815,2.848348,5.207846,-25.309827,41.628892,0.564297,0.518911,0.440242,0.965071,3.030816,1.663799,0.521936,-6.505907e-12,5.6,2.400000,5.600000,0.034934,0.709936,0.030257
3,soft_low_energy,600,0.957533,1.519106,0.037858,4.552722,2.574363,6.300617,-25.895373,51.540645,0.551667,0.510000,0.445000,0.881015,4.690721,0.993165,0.518333,1.600000e+00,0.8,3.200000,2.452804,0.063411,1.381942,0.013333


,metric,median,mean,p05,p25,p75,p95
0,model_based_uplift_pct = recommended_pred / cu...,2.244168,2.938223,0.146377,0.989080,4.070482,8.118032
1,predicted_regret_vs_operator_pct = recommended...,4.670562,7.930803,-24.284943,-7.612300,19.771873,50.227729
2,current_prediction_error_vs_operator_pct = cur...,1.695947,4.850874,-26.695159,-9.884113,16.433539,45.464430
3,bias_adjusted_regret_vs_operator_pct = regret_...,2.286845,3.079929,0.147710,1.009371,4.212935,8.858351


,metric,value
0,recommended_win_vs_operator_rate,0.601200
1,recommended_win_vs_operator_2pct_rate,0.558400
2,recommended_win_vs_operator_5pct_rate,0.491200
3,same_direction_axis_rate,0.090950
4,same_direction_rotation_rate,0.489583
5,same_direction_both_rate,0.098800


## 10. Example and save simulator-ready artifacts

In [11]:
example_row = test_df.sample(1, random_state=RANDOM_STATE + 10).iloc[0]
rec = recommend_for_row(example_row, grid_size=31)

example_actual_target_speed = float(example_row[target_speed])
example_current_pred_speed = float(rec["current_predicted_target_speed"])
example_recommended_pred_speed = float(rec["predicted_target_speed"])
example_predicted_regret_vs_operator_pct = 100.0 * (
    example_recommended_pred_speed / (example_actual_target_speed + EPS) - 1.0
)
example_current_error_vs_operator_pct = 100.0 * (
    example_current_pred_speed / (example_actual_target_speed + EPS) - 1.0
)
example_bias_adjusted_regret_pct = (
    example_predicted_regret_vs_operator_pct - example_current_error_vs_operator_pct
)

example_summary = pd.DataFrame([
    {
        "variant": "operator/current",
        "pressure_axis": example_row["pressure_axis"],
        "pressure_rotation": example_row["pressure_rotation"],
        "current_speed": example_row["speed"],
        "target_speed_actual": example_actual_target_speed,
        "predicted_target_speed": example_current_pred_speed,
        "predicted_uplift_pct": 0.0,
        "predicted_regret_vs_operator_pct": example_current_error_vs_operator_pct,
        "current_prediction_error_vs_operator_pct": example_current_error_vs_operator_pct,
        "bias_adjusted_regret_vs_operator_pct": 0.0,
        "delta_axis_pct": 0.0,
        "delta_rot_pct": 0.0,
    },
    {
        "variant": FINAL_OPTIMIZER_MODE,
        "pressure_axis": rec["recommended_pressure_axis"],
        "pressure_rotation": rec["recommended_pressure_rotation"],
        "current_speed": example_row["speed"],
        "target_speed_actual": example_actual_target_speed,
        "predicted_target_speed": example_recommended_pred_speed,
        "optimizer_score": rec["optimizer_score"],
        "predicted_uplift_pct": rec["predicted_uplift_pct"],
        "score_uplift_pct": rec["score_uplift_pct"],
        "predicted_regret_vs_operator_pct": example_predicted_regret_vs_operator_pct,
        "current_prediction_error_vs_operator_pct": example_current_error_vs_operator_pct,
        "bias_adjusted_regret_vs_operator_pct": example_bias_adjusted_regret_pct,
        "delta_axis_pct": rec["delta_pressure_axis_pct"],
        "delta_rot_pct": rec["delta_pressure_rotation_pct"],
    },
])

print("Example well:", example_row["well_id"])
print("Example time:", example_row["processing_time"])
print("Energy type:", example_row["rock_energy_type_final"])
display(example_summary)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(rotation_model, ARTIFACT_DIR / "rotation_model_near5.joblib")
joblib.dump(speed_model, ARTIFACT_DIR / "speed_model_near5.joblib")

with open(ARTIFACT_DIR / "surface_ranges_by_energy_type.json", "w", encoding="utf-8") as f:
    json.dump(surface_ranges, f, ensure_ascii=False, indent=2)

feature_config = {
    "data_path": str(DATA_PATH),
    "rock_energy_segmentation_expected_method": "energy_type_segment_quantile_log_pseudo_mse_only",
    "target_horizon": int(TARGET_HORIZON),
    "target_rotation": target_rotation,
    "target_speed": target_speed,
    "operator_comparison_target_pressure_axis": target_pressure_axis,
    "operator_comparison_target_pressure_rotation": target_pressure_rotation,
    "numeric_features": base_numeric_features,
    "base_numeric_features": base_numeric_features,
    "categorical_features": categorical_features,
    "rotation_features": base_numeric_features + categorical_features,
    "speed_features": speed_numeric_features + categorical_features,
    "speed_numeric_features": speed_numeric_features,
    "speed_extra_features": speed_extra_features,
    "removed_pruned_features": PRUNED_FEATURES_REMOVED,
    "feature_pruning_mode": "aggressive_correlation_and_low_importance_pruning",
    "hardness_feature_columns": HARDNESS_FEATURE_COLUMNS,
    "hardness_feature_policy": "single stable log-pseudo-MSE hardness feature; energy class based on 60-window segment quantiles",
    "energy_type_column": "rock_energy_type_final",
    "required_live_columns": [
        "processing_time", "well_id", "pressure_axis", "pressure_rotation",
        "rotation", "speed", "hardness_score_smooth", "rock_energy_type_final",
    ],
}

with open(ARTIFACT_DIR / "feature_config.json", "w", encoding="utf-8") as f:
    json.dump(feature_config, f, ensure_ascii=False, indent=2)

optimizer_config = {
    "optimizer_mode": FINAL_OPTIMIZER_MODE,
    "grid_size_default": int(GRID_SIZE),
    "max_delta_frac_default": float(MAX_DELTA_FRAC),
    "change_penalty_weight": float(CHANGE_PENALTY_WEIGHT),
    "boundary_penalty_weight": float(BOUNDARY_PENALTY_WEIGHT),
    "boundary_start": float(BOUNDARY_START),
    "score_formula": "pred_target_speed - change_penalty - boundary_penalty",
    "uplift_formula_model_based": "100 * (recommended_predicted_target_speed / current_predicted_target_speed - 1)",
    "predicted_regret_vs_operator_formula": "100 * (recommended_predicted_target_speed / actual_operator_target_speed_near5 - 1)",
    "recommended_win_vs_operator_formula": "recommended_predicted_target_speed > actual_operator_target_speed_near5 * (1 + margin)",
    "current_prediction_error_formula": "100 * (current_predicted_target_speed / actual_operator_target_speed_near5 - 1)",
    "bias_adjusted_regret_formula": "predicted_regret_vs_operator_pct - current_prediction_error_vs_operator_pct",
    "direction_agreement_formula": "sign(recommended_delta_pressure_pct) == sign(operator_future_delta_pressure_pct), with a 0.5 pp deadband",
    "operator_comparison_interpretation": "offline counterfactual diagnostics; not a measured causal effect because the recommendation was not actually applied",
    "use_local_reachable_bounds": True,
    "use_energy_type_quantile_bounds": True,
}

with open(ARTIFACT_DIR / "optimizer_config.json", "w", encoding="utf-8") as f:
    json.dump(optimizer_config, f, ensure_ascii=False, indent=2)

training_report = {
    "model_type": "LightGBM LGBMRegressor",
    "rock_energy_segmentation_expected_method": "energy_type_segment_quantile_log_pseudo_mse_only",
    "rotation_model_near5": rotation_metrics,
    "speed_model_oracle_rotation_near5": speed_metrics_oracle,
    "speed_model_chained_near5": speed_metrics_chained,
    "baseline_compare": baseline_compare.to_dict(orient="records"),
    "optimizer_summary": optimizer_summary.to_dict(orient="records"),
    "uplift_by_energy_type": by_energy.to_dict(orient="records"),
    "operator_comparison_summary": operator_comparison_summary.to_dict(orient="records"),
    "win_rate_summary": win_rate_summary.to_dict(orient="records"),
    # Backward-compatible key from the previous notebook version.
    "uplift_compare": uplift_compare.to_dict(orient="records"),
    "final_optimizer_mode": FINAL_OPTIMIZER_MODE,
    "feature_pruning_mode": "aggressive_correlation_and_low_importance_pruning",
    "removed_pruned_features": PRUNED_FEATURES_REMOVED,
    "hardness_feature_columns": HARDNESS_FEATURE_COLUMNS,
}

with open(ARTIFACT_DIR / "training_report.json", "w", encoding="utf-8") as f:
    json.dump(training_report, f, ensure_ascii=False, indent=2)

rec_df.to_csv(ARTIFACT_DIR / "offline_recommendations_light_penalty.csv", index=False)
optimizer_summary.to_csv(ARTIFACT_DIR / "optimizer_summary.csv", index=False)
by_energy.to_csv(ARTIFACT_DIR / "uplift_by_energy_type.csv", index=False)
operator_comparison_summary.to_csv(ARTIFACT_DIR / "operator_comparison_summary.csv", index=False)
win_rate_summary.to_csv(ARTIFACT_DIR / "win_rate_summary.csv", index=False)
uplift_compare.to_csv(ARTIFACT_DIR / "uplift_compare.csv", index=False)

metrics_summary = pd.DataFrame([
    {"model": "rotation_model_near5", **rotation_metrics},
    {"model": "speed_model_oracle_rotation_near5", **speed_metrics_oracle},
    {"model": "speed_model_chained_near5", **speed_metrics_chained},
])
metrics_summary.to_csv(ARTIFACT_DIR / "training_metrics.csv", index=False)

rec_df.to_csv(REPORT_DIR / "offline_recommendations_light_penalty.csv", index=False)
optimizer_summary.to_csv(REPORT_DIR / "optimizer_summary.csv", index=False)
by_energy.to_csv(REPORT_DIR / "uplift_by_energy_type.csv", index=False)
operator_comparison_summary.to_csv(REPORT_DIR / "operator_comparison_summary.csv", index=False)
win_rate_summary.to_csv(REPORT_DIR / "win_rate_summary.csv", index=False)
uplift_compare.to_csv(REPORT_DIR / "uplift_compare.csv", index=False)
metrics_summary.to_csv(REPORT_DIR / "training_metrics.csv", index=False)

print("Saved artifacts to:", ARTIFACT_DIR.resolve())
for p in sorted(ARTIFACT_DIR.iterdir()):
    print(" -", p.name)


Example well: 26435
Example time: 2025-10-15 00:45:28.008000
Energy type: hard_high_energy


,variant,pressure_axis,pressure_rotation,current_speed,target_speed_actual,predicted_target_speed,predicted_uplift_pct,predicted_regret_vs_operator_pct,current_prediction_error_vs_operator_pct,bias_adjusted_regret_vs_operator_pct,delta_axis_pct,delta_rot_pct,optimizer_score,score_uplift_pct
0,operator/current,19981.000000,15348.000,0.01818,0.01313,0.014448,0.00000,10.039645,10.039645,0.000000,0.000000,0.0,NaN,NaN
1,light_penalty,19874.434667,16084.704,0.01818,0.01313,0.014605,1.08181,11.230072,10.039645,1.190427,-0.533333,4.8,0.014578,0.899587


Saved artifacts to: /home/alex/Desktop/OptimalDrilling/notebooks/drilling_advisory_light_penalty_artifacts
 - feature_config.json
 - offline_recommendations_light_penalty.csv
 - operator_comparison_summary.csv
 - optimizer_config.json
 - optimizer_summary.csv
 - rotation_model_near5.joblib
 - speed_model_near5.joblib
 - surface_ranges_by_energy_type.json
 - training_metrics.csv
 - training_report.json
 - uplift_by_energy_type.csv
 - uplift_compare.csv
 - win_rate_summary.csv


## What simulator should load

Для подключения к симулятору нужны:

```text
drilling_advisory_light_penalty_artifacts/
    rotation_model_near5.joblib
    speed_model_near5.joblib
    feature_config.json
    optimizer_config.json
    surface_ranges_by_energy_type.json
```

В online/replay режиме симулятор должен хранить rolling buffer telemetry, создавать те же признаки, строить candidate grid `p_ax/p_rot`, считать `target_rotation_near5`, `target_speed_near5`, `optimizer_score`, выбирать recommended point и рисовать current point + recommended point на эмпирической surface текущего уровня энергоёмкости.

Offline-отчёты также сохраняют `operator_comparison_summary.csv` и `win_rate_summary.csv`. Эти файлы нужны только для анализа качества рекомендаций в notebook и не являются обязательными для работы симулятора.
